## Introdução e Contexto:

A empresa Megaline quer desenvolver um modelo que possa analisar o comportamento do cliente e recomendar um dos planos mais recentes da Megaline: Smart ou Ultra.

Nosso conjunto de dados será os dados de comportamento dos **clientes que já mudaram para os novos planos**, com a feature "is_ultra" indicando se o plano atual é o Ultra (1), ou Smart(0).

O objetivo central é escolher qual o modelo capaz de executar a tarefa de classificação com a maior acurácia possível (neste projeto, o limite para acurácia é 0,75) e faremos a verificação da acurácia com o conjunto de dados de teste.

## Preparação do ambiente

### Importe as bibliotecas

In [42]:
# Configuração inicial
import pandas as pd
#sklearn
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

### Definição das configurações de visualização.

In [43]:
# Variável Global para aleatoriedade
SEED = 42

# Ajuste das quantidade das colunas exibidas
pd.set_option('display.max_columns', 100)

### Leitura dos Dados

In [44]:
df = pd.read_csv("users_behavior.csv")

print(df.head())
print('-'*50)
print(df.info())
print('-'*50)
print(df["is_ultra"].value_counts(normalize=True))
print('-'*50)
print(df.describe())
print('-'*50)
print(f"Total de Duplicadas: {df.duplicated().sum()}")

   calls  minutes  messages   mb_used  is_ultra
0   40.0   311.90      83.0  19915.42         0
1   85.0   516.75      56.0  22696.96         0
2   77.0   467.66      86.0  21060.45         0
3  106.0   745.53      81.0   8437.39         1
4   66.0   418.74       1.0  14502.75         0
--------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 3214 entries, 0 to 3213
Data columns (total 5 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   calls     3214 non-null   float64
 1   minutes   3214 non-null   float64
 2   messages  3214 non-null   float64
 3   mb_used   3214 non-null   float64
 4   is_ultra  3214 non-null   int64  
dtypes: float64(4), int64(1)
memory usage: 125.7 KB
None
--------------------------------------------------
is_ultra
0    0.693528
1    0.306472
Name: proportion, dtype: float64
--------------------------------------------------
             calls      minutes     messages       mb_used     i

Como vimos acima, aparentemente não temos problemas com dados ausentes porque o `DataFrame` já foi preparado para o treinamento dos modelos.

Vale ressaltar neste ponto do projeto que a presença de usuários com plano ULTRA é 30%, então o desbalanceamento das classes neste problema de classificação é na razão de 70-30, mas o desbalanceamento não impede o treinamento.

## Preparação dos dados
Como o conjunto de dados de teste não existe, vamos dividir o conjunto de dados em Treinamento-Validação-Teste na proporção 3:1:1, isto é, 60% dos dados como Treinamento, 20% como validação, e 20% como dados de testes para as verificações finais. 

Vale ressaltar também, que já dividiremos de antemão as variáveis `features` e `target`, sendo a variável objetivo apenas a classificação `"is_ultra"` e as demais características permanecerão em features 

### Divisão das variáveis Feature-Target:

In [45]:
#Divisão do df features e objeto series target
features = df.drop(['is_ultra'], axis=1)
target = df['is_ultra']

### Divisão dos dados Treinamento-Validação:

In [46]:
#Atribuição Múltipla dividindo 60-40 inicialmente
features_train, features_midle, target_train, target_midle = train_test_split(
    features,target, test_size = 0.4, random_state=SEED,stratify=target
)

### Divisão dos dados Validação-Teste:

In [47]:
#Atribuição Múltipla dividindo agora: 20% validação-20% teste
features_test, features_valid, target_test, target_valid = train_test_split(
    features_midle,target_midle, test_size = 0.5, random_state=SEED,stratify=target_midle
)

### Verificação das Proporções:

In [48]:
#Relatório final
print("Dados das features:")
print(f"Features de Treinamento: {len(features_train)*100 / len(df):.2f}% - Formato:{features_train.shape}")
print(f"Features de Validação: {len(features_valid)*100 / len(df):.2f}% - Formato:{features_valid.shape}")
print(f"Features de Teste: {len(features_test)*100 / len(df):.2f}% - Formato:{features_test.shape}")
print("\nDados de target:")
print(f"Target do Treinamento: {len(target_train)*100 / len(df):.2f}% - Formato:{target_train.shape}")
print(f"Target do Validação: {len(target_valid)*100 / len(df):.2f}% - Formato:{target_valid.shape}")
print(f"Target do Teste: {len(target_test)*100 / len(df):.2f}% - Formato:{target_test.shape}")

Dados das features:
Features de Treinamento: 59.99% - Formato:(1928, 4)
Features de Validação: 20.01% - Formato:(643, 4)
Features de Teste: 20.01% - Formato:(643, 4)

Dados de target:
Target do Treinamento: 59.99% - Formato:(1928,)
Target do Validação: 20.01% - Formato:(643,)
Target do Teste: 20.01% - Formato:(643,)


## Modelos para tarefa de Classificação:

Vamos a partir de agora nos concentrar em criar, treinar e avaliar como cada um dos modelos se comporta em relação ao `conjunto de dados de validação`. 
Para isso, testaremos 3 vias que se encaixam com a proposta de solução para o problema apresentado:

- `DecisionTreeClassifier`: árvore de decisão única com `max_depth`, `min_samples_split` e `min_samples_leaf` ajustáveis.

- `RandomForestClassifier`: floresta com árvores de decisão aleatórias, com `max_depth` e `n_estimators` ajustáveis.

- `LogisticRegression`: regressão a uma curva sigmoide com `solver` e `class_weight` ajustáveis.

Três informações muito importantes devem ser reforçadas neste momento:

1. Todos os modelos serão treinados com os mesmos dados, para comparação justa.
2. Todos os modelos terão a mesma semente de aleatoriedade (SEED)
3. Neste caso, a métrica **ACURÁCIA** é o nosso **key_result** porque a classificação correta das duas classes é importante para a Megaline.


### DecisionTrees:

Para escolher o melhor modelo de árvore de decisão vamos criar ciclos de repetição alternando os hiperparâmetros para escolher o que possui a maior acurácia no conjunto de dados de validação. Escolhemos:
- `max_depth`: inteiro variando de 1 a 7, define a profundidade máxima de cada árvore

- `criterion`: será o de **gini** medindo a pureza de classe em cada nó, como o padrão do modelo

- `min_samples_split`: representa o número mínimo de observações que devem estar presentes em um nó antes que ele se divida. Testaremos com tipo float representando a fração de 1% a 5%, segundo a documentação oficial ele considerará o teto(min_samples_split * n_samples), se adaptando melhor ao tamanho do conjunto de dados de treinamento. De semelhante também testaremos parâmetros inteiros de 2 a 10, que neste caso serão mais permissivos.

- `min_samples_leaf`: representa o número mínimo de observações que um nó final (folha) deve ter. Testaremos de forma análoga ao feito acima.

In [49]:
#Criando Loop para testar variações apenas em max_depth 1-7
# Definindo valores iniciais
best_tree_accuracy = 0
best_tree_depth = 0

for depth in range(1,8):
    #Iniciando o modelo:
    model_tree = DecisionTreeClassifier(random_state=SEED, max_depth=depth)
    model_tree.fit(features_train, target_train)
    train_score = model_tree.score(features_train, target_train) 
    valid_score = model_tree.score(features_valid, target_valid)
    print(f"Modelo com `max_depth`={depth}:\nAcurácia nos dados de TREINAMENTO:{train_score*100:.6f}%\nAcurácia nos dados de VALIDAÇÃO:{valid_score*100:.6f}%")
    print('-'*50)
    if valid_score > best_tree_accuracy:
        best_tree_accuracy = valid_score
        best_tree_depth = depth

print('='*50)
print(f"Melhor Acurácia em modelos treinados apenas com 'max_depth':\nAcurácia={best_tree_accuracy*100.:6f}%\nMelhor Profundidade de Árvore={best_tree_depth}")


Modelo com `max_depth`=1:
Acurácia nos dados de TREINAMENTO:75.103734%
Acurácia nos dados de VALIDAÇÃO:74.961120%
--------------------------------------------------
Modelo com `max_depth`=2:
Acurácia nos dados de TREINAMENTO:78.526971%
Acurácia nos dados de VALIDAÇÃO:77.449456%
--------------------------------------------------
Modelo com `max_depth`=3:
Acurácia nos dados de TREINAMENTO:79.771784%
Acurácia nos dados de VALIDAÇÃO:78.382582%
--------------------------------------------------
Modelo com `max_depth`=4:
Acurácia nos dados de TREINAMENTO:79.875519%
Acurácia nos dados de VALIDAÇÃO:79.004666%
--------------------------------------------------
Modelo com `max_depth`=5:
Acurácia nos dados de TREINAMENTO:81.690871%
Acurácia nos dados de VALIDAÇÃO:79.315708%
--------------------------------------------------
Modelo com `max_depth`=6:
Acurácia nos dados de TREINAMENTO:82.468880%
Acurácia nos dados de VALIDAÇÃO:78.382582%
--------------------------------------------------
Modelo com

Notamos que o a maior acurácia foi obtida com `max_depth` = 7, chegando a 79.471229%.

Será que o modelo performaria bem sem sugerir parâmetro de profundidade máxima? Vamos verificar com a configuração padrão da documentação da scikit-learn onde `max-depth`=`None`

In [50]:
#Testando apenas a função padrão da scikit-learn
#Iniciando o modelo:
model_tree = DecisionTreeClassifier(random_state=SEED, max_depth=None)
model_tree.fit(features_train, target_train)
train_score = model_tree.score(features_train, target_train) 
valid_score = model_tree.score(features_valid, target_valid)
print(f"Modelo com `max_depth`=None:\nAcurácia nos dados de TREINAMENTO:{train_score*100:.6f}%\nAcurácia nos dados de VALIDAÇÃO:{valid_score*100:.6f}%")
print('-'*50)
if valid_score > best_tree_accuracy:
    best_tree_accuracy = valid_score
    best_tree_depth = None

print('='*50)
print(f"Melhor Acurácia em modelos treinados apenas com 'max_depth':\nAcurácia={best_tree_accuracy*100.:6f}%\nMelhor Profundidade de Árvore={best_tree_depth}")

Modelo com `max_depth`=None:
Acurácia nos dados de TREINAMENTO:100.000000%
Acurácia nos dados de VALIDAÇÃO:72.317263%
--------------------------------------------------
Melhor Acurácia em modelos treinados apenas com 'max_depth':
Acurácia=79.471229%
Melhor Profundidade de Árvore=7


A configuração de profundidade ilimitada (`max_depth`=None) resultou em um modelo com acurácia 100% nos dados de treinamento e com a acurácia caindo drasticamente para 72.317263% nos dados de validação - apresentando uma evidência de overfitting.

**Portanto nosso melhor resultado de acurácia nas árvores de decisão segue sendo na configuração `max_depth`=7.**

Vamos tentar criando ciclos de repetição com novos valores para os outros hiperparâmetros `min_samples_split` e `min_samples_leaf`, além de aumentar o range dos valores de `max_depth` para testat de 1-10, já que o melhor valor atingiu o máximo do último intervalo. 

In [76]:
#Criando Loop para testar os valores tipo INT
best_tree_accuracy = 0
best_tree_train_accuracy = 0
best_tree_depth = 0
best_tree_mss = 0
best_tree_msl = 0

for depth in range(1,11):
    for min_split in range(2,11):
        for min_leaf in range(1,11):
            #Iniciando o modelo:
            model_tree = DecisionTreeClassifier(random_state=SEED, max_depth=depth, min_samples_split=min_split, min_samples_leaf=min_leaf)
            model_tree.fit(features_train, target_train)
            train_score = model_tree.score(features_train, target_train) 
            valid_score = model_tree.score(features_valid, target_valid)
            if valid_score > best_tree_accuracy:
                best_tree_accuracy = valid_score
                best_tree_train_accuracy = train_score
                best_tree_depth = depth
                best_tree_mss = min_split
                best_tree_msl = min_leaf

print('='*80)
print(f"Melhor Acurácia das DecisionTrees com 'max_depth', 'min_samples_split' e 'min_samples_leaf' INTEIROS:\nAcurácia nos dados de TREINAMENTO:{best_tree_train_accuracy*100.:6f}%\nAcurácia na VALIDAÇÃO={best_tree_accuracy*100.:6f}%\nMelhor Profundidade de Árvore={best_tree_depth}\nMelhor min_samples_split={best_tree_mss}\nMelhor min_samples_leaf={best_tree_msl}")



Melhor Acurácia das DecisionTrees com 'max_depth', 'min_samples_split' e 'min_samples_leaf' INTEIROS:
Acurácia nos dados de TREINAMENTO:83.817427%
Acurácia na VALIDAÇÃO=79.782271%
Melhor Profundidade de Árvore=8
Melhor min_samples_split=2
Melhor min_samples_leaf=8


Adicionando os dois hiperparâmetros inteiros e aumentando o intervalo para testar o parâmetro max_depth, conseguimos atingir uma acurácia um pouco maior - 79.782271% - com os parâmetros **max_depth=8, min_samples_split=2, e min_samples_leaf=8**.

Vamos verificar como o modelo se comporta ao realizar testes entre 1% e 5% para estes valores.

In [78]:
#Criando Loop para testar os valores tipo FLOAT
best_tree_accuracy = 0
best_tree_train_accuracy = 0
best_tree_depth = 0
best_tree_mss = 0
best_tree_msl = 0

for depth in range(1,11):
    for min_split in range(1,6):
        for min_leaf in range(1,6):
            #Iniciando o modelo:
            model_tree = DecisionTreeClassifier(random_state=SEED, max_depth=depth, min_samples_split=min_split*0.01, min_samples_leaf=min_leaf*0.01)
            model_tree.fit(features_train, target_train)
            train_score = model_tree.score(features_train, target_train) 
            valid_score = model_tree.score(features_valid, target_valid)
            if valid_score > best_tree_accuracy:
                best_tree_accuracy = valid_score
                best_tree_train_accuracy = train_score
                best_tree_depth = depth
                best_tree_mss = min_split*0.01
                best_tree_msl = min_leaf*0.01

print('='*80)
print(f"Melhor Acurácia das DecisionTrees com 'max_depth' INTEIRO, 'min_samples_split' e 'min_samples_leaf' FLOAT:\nAcurácia nos dados de TREINAMENTO:{best_tree_train_accuracy*100.:6f}%\nAcurácia={best_tree_accuracy*100.:6f}%\nMelhor Profundidade de Árvore={best_tree_depth}\nMelhor min_samples_split={best_tree_mss}\nMelhor min_samples_leaf={best_tree_msl}")



Melhor Acurácia das DecisionTrees com 'max_depth' INTEIRO, 'min_samples_split' e 'min_samples_leaf' FLOAT:
Acurácia nos dados de TREINAMENTO:81.431535%
Acurácia=79.004666%
Melhor Profundidade de Árvore=5
Melhor min_samples_split=0.01
Melhor min_samples_leaf=0.01


Nesta configuração, os parâmetros mais permissivos foram os que performaram melhor, mas vale ressaltar que a acurácia não foi maior do que a acurácia medida e validada anteriormente de 79.782271%.

Portanto, para as **Árvores de Decisão**, nosso melhor resultado foi:
- **Melhor**
 `Acurácia`: 79.782271%
- `max_depth`=8
- `min_samples_split`=2
- `min_samples_leaf`=8

### RandomForest:

Para escolher o melhor modelo de `RandomForest` além da profundidade de cada árvore, deveremos escolher também o número ideal para o parâmetro `n_estimators` que será o número de árvores na floresta aleatória que o modelo será treinado. Manteremos os mesmos critérios que anteriormente para os demais parâmetros além do número dos estimadores. 

Escolhemos:
- `n_estimators`: Vamos aumentando o número de estimadores de 10 em 10, até 100. 

- `max_depth`: inteiro variando de 1 a 7, define a profundidade máxima de cada árvore

- `criterion`: será o de **gini** medindo a pureza de classe em cada nó exatamente como anteriormente, como o padrão do modelo


Começaremos testando apenas os modelos com número de estimadores.

In [81]:
#Criando Loop para testar apenas os valores de n_estimators
best_rf_accuracy = 0
best_rf_train_accuracy = 0
best_rf_estimators = 0

for est in range(10,101,10):
    #Iniciando o modelo:
    model_rf = RandomForestClassifier(random_state=SEED, n_estimators=est)
    model_rf.fit(features_train, target_train)
    train_score = model_rf.score(features_train, target_train) 
    valid_score = model_rf.score(features_valid, target_valid)
    if valid_score > best_rf_accuracy:
        best_rf_accuracy = valid_score
        best_rf_train_accuracy = train_score
        best_rf_estimators = est

print('='*80)
print(f"Melhor Acurácia das RandomForest apenas com 'n_estimators':\nAcurácia nos dados de TREINAMENTO:{best_rf_train_accuracy*100.:6f}%\nAcurácia na VALIDAÇÃO={best_rf_accuracy*100.:6f}%\nMelhor Número de Estimadores={best_rf_estimators}")



Melhor Acurácia das RandomForest apenas com 'n_estimators':
Acurácia nos dados de TREINAMENTO:99.896266%
Acurácia na VALIDAÇÃO=80.248834%
Melhor Número de Estimadores=50


Nesta primeira verificação, o melhor número de estimadores é 50.

Realizaremos novos testes com ciclos de repetição alterando também o outro hiperparâmetro `max_depth`. 

É importante reforçar que escolhemos não incluir os hiperparâmetros `min_samples_split` e `min_samples_leaf` por economia de tempo e recursos computacionais, porque já conseguimos atingir uma acurácia maior que as Árvores de Decisão da sessão anterior apenas com o hiperparâmetro `n_estimators`.

In [82]:
#Criando Loop para testar combinações de n_estimators com max_depth
best_rf_accuracy = 0
best_rf_train_accuracy = 0
best_rf_estimators = 0
best_rf_depth = 0

for est in range(10,101,10):
    for depth in range(1,8):
        #Iniciando o modelo:
        model_rf = RandomForestClassifier(random_state=SEED, n_estimators=est, max_depth=depth)
        model_rf.fit(features_train, target_train)
        train_score = model_rf.score(features_train, target_train) 
        valid_score = model_rf.score(features_valid, target_valid)
        if valid_score > best_rf_accuracy:
            best_rf_accuracy = valid_score
            best_rf_train_accuracy = train_score
            best_rf_estimators = est
            best_rf_depth = depth

print('='*80)
print(f"Melhor Acurácia das RandomForest alternando hiperparâmetros:\nAcurácia nos dados de TREINAMENTO:{best_rf_train_accuracy*100.:6f}%\nAcurácia na VALIDAÇÃO={best_rf_accuracy*100.:6f}%\nMelhor Número de Estimadores={best_rf_estimators}\nmax_depth={best_rf_depth}")




Melhor Acurácia das RandomForest alternando hiperparâmetros:
Acurácia nos dados de TREINAMENTO:83.454357%
Acurácia na VALIDAÇÃO=81.181960%
Melhor Número de Estimadores=10
max_depth=6


No último teste encontramos uma combinação de hiperparâmetros para o modelo que performou até melhor nos dados de validação do que apenas utilizando o número de estimadore - e ainda com uma diferença de overfitting menor (lacuna entre treino e validação).

Portanto, para as **Florestas Aleatórias**, nosso melhor resultado foi:
- **Melhor** `Acurácia`=81.181960%
- Melhor Número de Estimadores=10
- max_depth=6

É importante observar que esta diferença de cerca de um ponto percentual para a segunda melhor `RandomForest` pode ter sido produzida pelo acaso, então vale a pena registrarmos aqui as duas melhores configurações e a diferença entre elas.

In [84]:
best_forest_parameters = [
    {'name': 'RandomForest', 'n_estimators': 10, 'max_depth': 6, 'train_accuracy': 0.83454357, 'validation_acuracy': 0.81181960},
    {'name': 'RandomForest', 'n_estimators': 50, 'max_depth': None, 'train_accuracy': 0.99896266, 'validation_acuracy': 0.80248834}
]

best_forests = pd.DataFrame(data=best_forest_parameters)
best_forests['overfitting_difference'] = best_forests['train_accuracy'] - best_forests['validation_acuracy']
best_forests

,name,n_estimators,max_depth,train_accuracy,validation_acuracy,overfitting_difference
0,RandomForest,10,6.0,0.834544,0.811820,0.022724
1,RandomForest,50,NaN,0.998963,0.802488,0.196474


### LogisticRegression:

Para encontrarmos a melhor curva de `LogisticRegression` vamos observar se o hiperparâmetro `class_weight` balanceado ajuda a termos melhores resultados ou não - já que já foi observado um desbalanceamento de 70-30 entre as classes.

Escolhemos:
- `solver`: testaremos `liblinear`, porque um teste já foi executado com 'lbfgs' padrão da biblioteca scikit-learn e houve uma mensagem de que o modelo atingiu o limite de 100 iterações antes de satisfazer o critério de convergência, e não queremos inserir o hiperparâmetro de `max_iter` que foi testado com valor 1000 e não obteve resultados satisfatórios.

- `class_weight`: além de testar o padrão da biblioteca scikit-learn que é `None`, vamos testar também `balanced` já que temos números diferentes de usuários em cada classe.


Começaremos testando apenas o modelo com os padrões da biblioteca, sem especificar `class_weight`:

In [68]:
#Criando teste para padrões de LogisticRegression scikit-learn
model_log = LogisticRegression(random_state=SEED, solver='liblinear')
model_log.fit(features_train,target_train)
train_score = model_log.score(features_train, target_train) 
valid_score = model_log.score(features_valid, target_valid)
print(f"Modelo LogisticRegression padrão:\nAcurácia nos dados de TREINAMENTO:{train_score*100:.6f}%\nAcurácia nos dados de VALIDAÇÃO:{valid_score*100:.6f}%")
print('-'*50)

Modelo LogisticRegression padrão:
Acurácia nos dados de TREINAMENTO:70.176349%
Acurácia nos dados de VALIDAÇÃO:69.828927%
--------------------------------------------------


In [69]:
#Criando Loop para testar as combinações dos hiperparâmetros acima
best_log_accuracy = 0
best_log_cw = ""

for class_type in [None, 'balanced']:
    model_log = LogisticRegression(random_state=SEED, solver='liblinear', class_weight=class_type)
    model_log.fit(features_train,target_train)
    train_score = model_log.score(features_train, target_train) 
    valid_score = model_log.score(features_valid, target_valid)
    print(f"Modelo LogisticRegression - `class_weight`={class_type} :\nAcurácia nos dados de TREINAMENTO:{train_score*100:.6f}%\nAcurácia nos dados de VALIDAÇÃO:{valid_score*100:.6f}%")
    print('-'*50)
    if valid_score > best_log_accuracy:
        best_log_accuracy = valid_score
        best_log_cw = class_type
        

print('='*80)
print(f"Melhor Acurácia da LogisticRegression:\nAcurácia={best_log_accuracy*100.:6f}%\nMelhor `class_weight`: {best_log_cw}")


Modelo LogisticRegression - `class_weight`=None :
Acurácia nos dados de TREINAMENTO:70.176349%
Acurácia nos dados de VALIDAÇÃO:69.828927%
--------------------------------------------------
Modelo LogisticRegression - `class_weight`=balanced :
Acurácia nos dados de TREINAMENTO:62.136929%
Acurácia nos dados de VALIDAÇÃO:64.385692%
--------------------------------------------------
Melhor Acurácia da LogisticRegression:
Acurácia=69.828927%
Melhor `class_weight`: None


Podemos observar que o nosso melhor resultado com o modelo de LogisticRegression foram:
- **Melhor** `Acurácia`=69.828927%
- Melhor `class_weight`: None

Ficando abaixo do nosso limite para acurácia de 75%. Sendo imediatamente descartado da nossa análise.

Uma observação relevante é que este modelo também seria reprovado na prova real onde todos são classificados como plano smart, como fizemos no final do caderno e também mostrou uma acurácia próxima a 69%.

### Conclusões da comparação entre os modelos:

Como mostramos com números, já descartamos de imediato o melhor modelo de regressão logística por não termos atingido o limite mínimo de acurácia de 75%.

Qual modelo devemos escolher entre os modelos de Árvore de Decisão (única) ou Floresta Aleatória? 

Naturalmente, como o objetivo da nossa análise é a maior acurácia possível, já é natural que nós esperemos uma maior acurácia de um modelo de RandomForest, porque ele combina os resultados de diversas árvores (estimadores) para nos dar a nossa melhor resposta. 

Essa resposta se confirmou com os números que traremos aqui abaixo:

#### Maior Acurácia das `DecisionTreeClassifier`:
- **Melhor**
 `Acurácia`: 79.782271%
- `max_depth`=8
- `min_samples_split`=2
- `min_samples_leaf`=8

#### Maior Acurácia das `RandomForest`:
- **Melhor** `Acurácia`=81.181960%
- Melhor Número de Estimadores=10
- max_depth=6


Portanto, o modelo que foi escolhido para realizar esta tarefa de classificação é o modelo de `RandomForest` com n_estimators = 10 e max_depth=6. Verificaremos se o resultado será mantido com o conjunto de dados de teste.

## Testando o modelo escolhido:

Nesta etapa, faremos a verificação da acurácia do modelo diante do conjunto de dados de teste, para entender se ele poderá ser ou não efetivamente aprovado.

Antes de verificar a acurácia do modelo no conjunto de dados de teste, faremos a prova real do nosso modelo com "modelos extremos" que classificam todos como 1 ou todos como 0 na coluna `is_ultra` para nos certificarmos que ele terá um desempenho **melhor que o acaso**.

In [85]:
# Acurácia de um modelo extremo caso classificasse todos como 1
#baseline para dados de validação - todos 1
all_as_one_valid = pd.Series(data=1, index=target_valid.index)
#baseline para dados de teste - todos 1
all_as_one_test = pd.Series(data=1, index=target_test.index)
valid_accuracy = accuracy_score(target_valid, all_as_one_valid)
test_accuracy = accuracy_score(target_test, all_as_one_test)
print(f"Acurácia do modelo extremo que classifica tudo como 1:\nnos dados de Validação:{valid_accuracy*100:.6f}%\nnos dados de Teste:{test_accuracy*100:.6f}%")
print('='*80)

# Acurácia de um modelo extremo caso classificasse todos como 0
#baseline para dados de validação - todos 0
all_as_zero_valid = pd.Series(data=0, index=target_valid.index)
#baseline para dados de teste - todos 0
all_as_zero_test = pd.Series(data=0, index=target_test.index)
valid_accuracy = accuracy_score(target_valid, all_as_zero_valid)
test_accuracy = accuracy_score(target_test, all_as_zero_test)
print(f"Acurácia do modelo extremo que classifica tudo como 0:\nnos dados de Validação:{valid_accuracy*100:.6f}%\nnos dados de Teste:{test_accuracy*100:.6f}%")

Acurácia do modelo extremo que classifica tudo como 1:
nos dados de Validação:30.637636%
nos dados de Teste:30.637636%
Acurácia do modelo extremo que classifica tudo como 0:
nos dados de Validação:69.362364%
nos dados de Teste:69.362364%


Nosso modelo escolhido supera o baseline trivial nos dados de validação (cerca de 69,36%), apresentando na etapa de validação uma acurácia de cerca de 81,18%.

Vejamos a sua aplicação ao conjunto de dados de teste dividido no início do caderno. 

Testaremos com `max_depth` = 6 e `n_estimators` = 10.


In [86]:
#Criando o modelo com os parâmetros previamente setados
model = RandomForestClassifier(random_state=SEED, n_estimators=10, max_depth=6)
model.fit(features_train, target_train)
classification = model.predict(features_test)
final_accuracy = accuracy_score(target_test, classification)

print(f"Acurácia do modelo no conjunto de dados de teste:\n{final_accuracy*100:.6f}%")

Acurácia do modelo no conjunto de dados de teste:
79.471229%


Podemos observar que diante do conjunto de dados de teste, o nosso modelo também supera o baseline trivial (também cerca de 69,36%) com uma acurácia de cerca de 79,47%, vencendo também o limite mínimo de acurácia proposto de 75%.

## Conclusão:
O modelo de regressão logística acabou abaixo do limiar de acurácia definido, e praticamente empatou com a classificação de um baseline trivial que marca todos como 0.

Apesar da Árvore de decisão ter apresentado um desempenho melhor, a sua acurácia máxima de 79,78%% foi superada pelo modelo de floresta aleatória apresentou acurácia máxima de 81.18% - performance de ambos diante do conjunto de dados de validação.

O melhor modelo para resolver esta tarefa de classificação e ajudar os clientes da Megaline a migrar para os novos planos é o modelo de `RandomForest` com n_estimators=10 e max_depth=6, e o objetivo de obter um modelo que executasse esta tarefa com acurácia acima de 0.75 foi atingido - esta configuração obteve acurácia de cerca de 79.47% diante do conjunto de dados de teste.

Esta acurácia também superou o baseline trivial de cerca de 69,36% no conjunto de dados de teste (classificando todos os registros de teste como 0, a classe majoritária). Isso torna evidente o ganho real que o modelo desempenha na acurácia da classificação.